## Imports


In [7]:
import copy
import logging
import os
from pathlib import Path
from typing import Any, Dict, List, Optional

import open_clip
import wandb

import hydra
import omegaconf
import pytorch_lightning as pl
import torch
from hydra import compose, initialize
from hydra.utils import instantiate
from lightning.pytorch import Callback
from omegaconf import DictConfig, ListConfig, OmegaConf
from torch.nn.utils import parameters_to_vector, vector_to_parameters

from nn_core.callbacks import NNTemplateCore
from nn_core.common import PROJECT_ROOT
from nn_core.common.utils import enforce_tags, seed_index_everything
from nn_core.model_logging import NNLogger
from nn_core.serialization import NNCheckpointIO

# Force the execution of __init__.py if this file is executed directly.
import mass  # noqa
from mass.data.datasets.registry import get_dataset
from mass.modules.encoder import ClassificationHead, ImageEncoder
from mass.modules.projection_router import ProjectionRouter
from mass.modules.nn_router import NNRouter
from mass.modules.heads import get_classification_head
from mass.modules.router import AbstractRouter
from mass.utils.io_utils import load_model_from_disk
from mass.utils.plots import plot_interactive_radar_chart
from mass.utils.utils import (
    compute_task_dict, 
    apply_dict_to_model,
    build_callbacks,
    get_finetuning_accuracies,
    add_normalized_accuracy,
    compute_avg_accuracy,
    print_memory,
    get_routing_weights,
    svd_key_from_layer
)
from mass.task_vectors.task_singular_vectors import *
import json
import os

pylogger = logging.getLogger(__name__)

torch.set_float32_matmul_precision("high")

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import hydra
from hydra import initialize, compose
from typing import Dict, List

hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path=str("../conf"), job_name="debug_mnist")
cfg = compose(config_name="static_merging")

In [10]:
# upperbound accuracies, used for logging the normalized accuracy
finetuned_accuracies: Dict[str, float] = get_finetuning_accuracies(
    cfg.misc.finetuned_accuracy_path
)

# only has vision encoder, no text transformer
zeroshot_encoder: ImageEncoder = load_model_from_disk(
    cfg.misc.pretrained_checkpoint, model_name=cfg.nn.module.encoder.model_name
)

finetuned_name = (
    lambda name: Path(cfg.misc.ckpt_path) / f"{name}Val" / "model.pt"
)
finetuned_models = {
    dataset: load_model_from_disk(
        finetuned_name(dataset), model_name=cfg.nn.module.encoder.model_name
    ).state_dict()
    for dataset in cfg.benchmark.datasets
}


2025-09-03 15:25:42 INFO     Loading model from disk                                        ]8;id=539453;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py\mass.utils.io_utils]8;;\:]8;id=266598;file:///media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py#107\107]8;;\
                             /media/donato/Extra-storage/Code/model-merging/resources/check                        
                             points//ViT-B-32/base/model.pt                                                        

FileNotFoundError: [Errno 2] No such file or directory: '/media/donato/Extra-storage/Code/model-merging/resources/checkpoints//ViT-B-32/base/model.pt'

## Load all datasets

In [ ]:
# for dataset_name in cfg.benchmark.datasets:
#     pylogger.info(f"Loading dataset: {dataset_name}")

#     dataset_cfg = OmegaConf.load(
#     PROJECT_ROOT / "conf" / "dataset" / f"{dataset_name}.yaml"
#     )

#     dataset = instantiate(
#         dataset_cfg, preprocess_fn=zeroshot_encoder.val_preprocess
#     )

## Upload all models to HF

In [11]:
from huggingface_hub import HfApi, create_repo, upload_folder

with open(f"{PROJECT_ROOT}/secrets.txt", "r") as f:
    hf_token = f.readline().strip()

for dataset in cfg.benchmark.datasets:

    repo_id = f"crisostomi/ViT-B-32-{dataset}"
    create_repo(repo_id, repo_type="model", private=False, exist_ok=True, token=hf_token)

    upload_folder(
        folder_path=f"{PROJECT_ROOT}/../resources/checkpoints/ViT-B-32/{dataset}Val/",
        repo_id=repo_id,
        repo_type="model",
        commit_message="Initial upload",
        token=hf_token
    )

model.pt: 100%|██████████| 454M/454M [03:15<00:00, 2.33MB/s]   
model.pt: 100%|██████████| 454M/454M [03:12<00:00, 2.35MB/s] 
model.pt: 100%|██████████| 454M/454M [03:13<00:00, 2.35MB/s] 
model.pt: 100%|██████████| 454M/454M [03:12<00:00, 2.35MB/s] 
model.pt: 100%|██████████| 454M/454M [03:20<00:00, 2.26MB/s] 
model.pt: 100%|██████████| 454M/454M [03:22<00:00, 2.24MB/s] 
model.pt: 100%|██████████| 454M/454M [03:19<00:00, 2.28MB/s] 
model.pt: 100%|██████████| 454M/454M [03:15<00:00, 2.32MB/s] 
model.pt: 100%|██████████| 454M/454M [03:15<00:00, 2.33MB/s] 
model.pt: 100%|██████████| 454M/454M [03:16<00:00, 2.30MB/s] 
model.pt: 100%|██████████| 454M/454M [03:13<00:00, 2.35MB/s] 
model.pt: 100%|██████████| 454M/454M [03:13<00:00, 2.35MB/s] 
model.pt: 100%|██████████| 454M/454M [03:13<00:00, 2.34MB/s] 
model.pt: 100%|██████████| 454M/454M [03:14<00:00, 2.33MB/s] 
model.pt: 100%|██████████| 454M/454M [03:14<00:00, 2.33MB/s] 
model.pt: 100%|██████████| 454M/454M [03:14<00:00, 2.33MB/s] 
model.

In [12]:

repo_id = f"crisostomi/ViT-B-32-base"
create_repo(repo_id, repo_type="model", private=False, exist_ok=True, token=hf_token)

upload_folder(
    folder_path=f"{PROJECT_ROOT}/../resources/checkpoints/ViT-B-32/base/",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Initial upload",
    token=hf_token
)

[autoreload of mass.utils.io_utils failed: Traceback (most recent call last):
  File "/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/media/donato/Extra-storage/Code/model-merging/mass/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/donato/.local/share/uv/python/cpython-3.11.8-linux-x86_64-gnu/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/media/donato/Extra-storage/Code/model-merging/mass/src/mass/utils/io_utils.py", line 19, in <module>
    from libraries.mergen

CommitInfo(commit_url='https://huggingface.co/crisostomi/ViT-B-32-base/commit/a87d40e16e2975ecbc427b5009e237479fbf152d', commit_message='Initial upload', commit_description='', oid='a87d40e16e2975ecbc427b5009e237479fbf152d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/crisostomi/ViT-B-32-base', endpoint='https://huggingface.co', repo_type='model', repo_id='crisostomi/ViT-B-32-base'), pr_revision=None, pr_num=None)